# Task 2 (v2): Build your first AI agent

In this notebook, you learn how to build AI agents using the Strands Agents framework. You start by creating a conversational agent, then add tools to make it more capable. You finally build a practical recipe assistant that can search the web for cooking information. You use Amazon Bedrock with the Amazon Nova Lite model to power your agents.

AI agents differ from traditional LLMs because they can take actions, use tools, and work toward goals autonomously. Rather than only responding to questions, agents can actually do things for you.

> **What is new in v2:** In the original notebook the recipe assistant only had a `websearch` tool. Web search returns just short snippets (roughly 100&ndash;400 characters per result), so the agent could never see a *complete* recipe &mdash; it had to fill in the details from its own training data. This version adds the missing second step of the retrieval pipeline: a **`fetch_webpage`** tool that downloads the actual recipe page and extracts its readable text. The agent now works in two steps: **search &rarr; fetch &rarr; answer**, so the recipe it returns is genuinely retrieved from the web.

#### Scenario
You work for AnyCompany, a growing technology company that wants to explore how AI agents can help automate everyday tasks. Your team is interested in learning how these agents could help with tasks like research and customer support. You are starting with the basics of building intelligent agents.

## Task 2.1: Environment setup

In this task, you set up your environment by installing the necessary packages to get started with building AI agents.

In [ ]:
# Install the Strands Agents framework and tools
# %pip install strands-agents strands-agents-tools

## Task 2.2: Create your first AI agent

In this task, you create an AI agent that can have conversations. This agent uses Amazon Bedrock with the Amazon Nova Lite model to understand and respond to your messages. The system prompt defines the desired behavior of your agent.

In [ ]:
import warnings
warnings.filterwarnings(action="ignore", message=r"datetime.datetime.utcnow") 

from strands import Agent

# Create your first AI agent
agent = Agent(callback_handler=None,
    model="amazon.nova-lite-v1:0",
    system_prompt="You are a helpful assistant that provides concise responses."
)

Next, you test your agent by sending it a message.

<i aria-hidden="true" class="fas fa-sticky-note" style="color:#563377"></i> **Note:** The Strands framework uses Amazon Bedrock with the Amazon Nova Lite model by default. This model supports conversational interactions and can be enhanced with tools.

In [ ]:
# Send a message to the agent
response = agent("Hello! Tell me a joke.")
print(response)

## Task 2.3: Add tools to your agent

In this task, you give your agent tools so it can do more than chat.

- You add a calculator tool which is provided by the Strands Agents SDK.

- You create a weather tool using the @tool decorator. 

<i aria-hidden="true" class="fas fa-sticky-note" style="color:#563377"></i> **Note:** The weather tool is a placeholder example and will always return ‘Sunny’

In [ ]:
from strands import Agent, tool
from strands_tools import calculator

# Create a weather tool
@tool
def weather():
    """Get current weather information"""
    return "Sunny and O degree Celsius"

# Create an agent with tools
agent_with_tools = Agent(callback_handler=None,
    model="amazon.nova-lite-v1:0",
    tools=[calculator, weather],
    system_prompt="You are a helpful assistant. You can do math calculations and use the weather tool to tell the weather."
)

# Test the agent with both tools in one query
# Note: This query requires both tools - the weather tool to get temperature in Celsius,
# and the calculator tool to convert from Celsius to Fahrenheit.
# The agent will automatically determine which tools to use and in what order.
response = agent_with_tools("What is the weather in Seattle in Fahrenheit?")
print(response)

Now explore how the agent analyze the question and decide to use only the calculator tool

In [ ]:
# Test with a math question that only needs the calculator tool
# The agent will analyze the question and decide to use only the calculator tool
math_query = "What is 25 * 4 + 18?"
print("=== Agent Chooses Calculator Tool ===")
print(f"Query: {math_query}")
response = agent_with_tools(math_query)
print(f"Response: {response}")

### Direct Tool Invocation

You can also call tools directly without going through the agent conversation. This is useful for testing or when you want to use a specific tool function.

**Specification for direct tool invocation:**
- Use `agent.tool.tool_name()` to call a specific tool directly
- Pass the required parameters as function arguments
- This bypasses the agent's natural language processing and tool selection logic
- Useful for programmatic access to tool functionality

In [ ]:
# Call the calculator tool directly
# Note: Direct tool invocation bypasses the agent's conversation flow.
# Use agent_with_tools.tool.calculator() to call the calculator tool directly
# without the agent deciding which tool to use. This is useful for testing
# specific tools or when you know exactly which tool function you need.
result = agent_with_tools.tool.calculator(expression="2 + 3 * 4")
print(f"Calculator result: {result}")

## Task 2.4: Configure logging

In this task, you set up logging to understand what your agent is doing behind the scenes. This helps you understand how the agent processes requests and uses tools.

The Strands framework uses Python's standard logging module to provide visibility into agent operations. You can configure different log levels to get more or less detail about what your agent is doing.

In [ ]:
import logging
from strands import Agent
import os

# Enable detailed logging to understand what the agent is doing
logging.getLogger("strands").setLevel(logging.INFO)

# Set up logging to write to both console and file
logging.basicConfig(
    format="%(asctime)s | %(levelname)s | %(name)s | %(message)s",
    level=logging.INFO,
    handlers=[
        logging.StreamHandler()  # Console output
    ]
)

# Create a logger
logger = logging.getLogger("agent_activity")

# Create an agent with logging enabled
logger.info("Creating new agent with Nova Lite model")
logged_agent = Agent(callback_handler=None,model="amazon.nova-lite-v1:0")

logger.info("Sending message to agent: 'Hello! How are you?'")
response = logged_agent("Hello! How are you?")
print(response)

## Task 2.5: Explore model configuration

In this task, you learn how to configure different models and settings for your agents. You can specify which AI model to use and adjust its behavior with parameters like temperature. Temperature controls how creative or consistent the responses are. Lower temperature values make responses more consistent and predictable, while higher values make them more creative and varied.

In [ ]:
from strands import Agent
from strands.models import BedrockModel

# Create a custom model configuration
custom_model = BedrockModel(
    model_id="amazon.nova-lite-v1:0",
    temperature=0.3  # Lower temperature = more consistent responses
)

# Create an agent with the custom model
custom_agent = Agent(callback_handler=None,model=custom_model)
print("Agent created successfully!")

## Task 2.6: Build a recipe assistant agent (v2: search *and* fetch)

In this task, you create a more practical agent that can help with cooking. This recipe assistant can search the web for recipes and cooking information, showing how agents can be useful in real-world scenarios.

**Why two tools?** A web search only returns short snippets (title, URL, and roughly 100&ndash;400 characters of preview text per result). That is enough to *find* a recipe page, but not enough to *read* the recipe. The v2 agent therefore uses a two-step retrieval pipeline:

1. **`websearch`** &mdash; find candidate recipe pages (returns titles, URLs, and snippets).
2. **`fetch_webpage`** &mdash; download the most promising URL and extract the full readable text, so the agent can quote the actual ingredient list and instructions from the page.

First, install the web search package that your recipe agent needs. The fetch tool uses `requests` and `beautifulsoup4`, which are installed alongside it.

In [ ]:
%pip install ddgs requests beautifulsoup4

Create a web search tool that your recipe agent can use to find recipe pages online. Note that each result contains an `href` field &mdash; the URL that the fetch tool downloads in the second step.

In [ ]:
from strands import Agent, tool
from ddgs import DDGS
import logging

# Set up logging
logging.getLogger("strands").setLevel(logging.INFO)

# Create a web search tool (step 1 of the retrieval pipeline)
@tool
def websearch(keywords: str, max_results: int = 3) -> str:
    """Search the web for information. Returns titles, URLs (href) and short
    text snippets. The snippets are only previews - to read a page, pass its
    'href' URL to the fetch_webpage tool.
    Args:
        keywords (str): What to search for
        max_results (int): How many results to return
    Returns:
        Search results as text (title, href, body snippet per result)
    """
    try:
        results = DDGS().text(keywords, max_results=max_results)
        return str(results) if results else "No results found."
    except Exception as e:
        return f"Search error: {e}"

print("Web search tool created successfully!")

Now create the **web fetch tool** (step 2 of the retrieval pipeline). It downloads a page, removes non-content markup (scripts, styles, navigation), and returns the readable text. The output is capped at 8,000 characters so a long page does not overflow the model's context window &mdash; for recipe pages this is comfortably enough to include the full ingredient list and instructions.

In [ ]:
import requests
from bs4 import BeautifulSoup

# Create a web fetch tool (step 2 of the retrieval pipeline)
@tool
def fetch_webpage(url: str, max_chars: int = 8000) -> str:
    """Download a web page and return its readable text content.
    Use this after websearch to read the full content of a result URL,
    for example to get the complete ingredient list and cooking
    instructions of a recipe instead of only a search snippet.
    Args:
        url (str): The URL of the page to download (use the 'href' from websearch results)
        max_chars (int): Maximum number of characters of page text to return
    Returns:
        The extracted page text, or an error message
    """
    headers = {
        # Some recipe sites block requests without a browser-like User-Agent
        "User-Agent": ("Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 "
                       "(KHTML, like Gecko) Chrome/124.0 Safari/537.36"),
        "Accept-Language": "en-US,en;q=0.9",
    }
    try:
        resp = requests.get(url, headers=headers, timeout=15)
        resp.raise_for_status()
        content_type = resp.headers.get("Content-Type", "")
        if "html" not in content_type and not resp.text.lstrip().startswith("<"):
            return f"Fetch error: unsupported content type '{content_type}' at {url}"

        soup = BeautifulSoup(resp.text, "lxml")
        # Drop non-content elements before extracting text
        for element in soup(["script", "style", "noscript", "header", "footer", "nav", "aside", "form", "iframe"]):
            element.decompose()

        # Extract text line by line and drop blank lines
        lines = (line.strip() for line in soup.get_text(separator="\n").splitlines())
        text = "\n".join(line for line in lines if line)

        if not text:
            return f"Fetch error: no readable text found at {url}"
        if len(text) > max_chars:
            text = text[:max_chars] + "\n[... page text truncated ...]"
        return f"Content of {url}:\n\n{text}"
    except requests.RequestException as e:
        return f"Fetch error for {url}: {e}"

print("Web fetch tool created successfully!")

Now create the recipe assistant agent with **both** tools. The system prompt explicitly instructs the agent to follow the two-step pipeline: search first, then fetch the best result and answer from the fetched page content.

In [ ]:

# Create the recipe assistant agent (v2: search + fetch)
recipe_agent = Agent(callback_handler=None,
    model="amazon.nova-lite-v1:0",
    system_prompt="""You are RecipeBot, a helpful cooking assistant.
    Help users find recipes and answer cooking questions.

    Always follow this two-step retrieval process for recipe requests:
    1. Use the websearch tool to find candidate recipe pages. The search
       results only contain short snippets - never answer from snippets alone.
    2. Pick the most promising result and pass its 'href' URL to the
       fetch_webpage tool to download the full recipe page. If a page fails
       to fetch or contains no recipe, try the next search result.

    Base your answer on the fetched page content: give the complete
    ingredient list with quantities and the full step-by-step instructions
    exactly as found on the page, and always cite the source URL.""",
    tools=[websearch, fetch_webpage]
)

print("Recipe assistant agent created successfully!")

Test your recipe assistant by asking it for cooking help. Watch the log output: the agent first calls `websearch`, then calls `fetch_webpage` on one of the result URLs, and only then writes its answer &mdash; now based on the *complete* recipe from the fetched page, including exact quantities and all instruction steps, with the source URL cited.

In [ ]:
# Test the recipe assistant
response = recipe_agent("Find a chicken and broccoli recipe online and give me the full ingredient list and instructions from the page.")
print(response)

You have successfully created a practical AI agent that can search the web, download the pages it finds, and help with cooking questions. This demonstrates a complete retrieval pipeline: the search step *finds* sources, and the fetch step *reads* them, so the agent's answer is grounded in real web content instead of short search snippets.

You have now experimented with the Strands Agents framework, which provides a way to build AI agents that can use tools and take actions. Using this framework, you have learned how agents differ from traditional LLMs by being able to actively use tools to accomplish tasks.

### Try it yourself
- Ask the agent to compare recipes from two different pages (it will need to call `fetch_webpage` twice).
- Modify the system prompts to create agents for different use cases.
- Create custom tools for specific tasks your team might need.
- Experiment with different model configurations to understand how they affect agent behavior.

### Cleanup

You have completed this notebook. To move to the next part of the lab, do the following:

- Close this notebook file and continue with the **Conclusion**.